# 🌿 Détection des Maladies des Plantes — Deep Learning
## ÉTAPE 4/9 : Fine-Tuning MobileNetV2 (Transfer Learning)

**Dataset :** PlantVillage — 54 305 images RGB (224×224×3), 38 classes  
**Objectif :** Surpasser le CNN From Scratch grâce au Transfer Learning sur MobileNetV2 pré-entraîné sur ImageNet.

---

### 📋 Plan du notebook

| Partie | Contenu |
|--------|---------|
| 0 | Reconstruction des générateurs (artefacts de l'étape 2) |
| 1 | Chargement de MobileNetV2 pré-entraîné |
| 2 | Construction du modèle complet (base + tête de classification) |
| 3 | Phase 1 — Feature Extraction (couches de base gelées, 5 epochs) |
| 4 | Phase 2 — Fine-Tuning (20 dernières couches dégelées, LR=1e-5, 10 epochs) |
| 5 | Évaluation (Accuracy, Loss, Precision, Recall, F1) |
| 6 | Visualisations (courbes, matrice de confusion, classification report) |
| 7 | Comparaison CNN From Scratch vs MobileNetV2 |
| 8 | Sauvegarde (`.keras` et `.h5`) |
| 9 | Analyse professionnelle complète |

> 💡 Chaque partie est précédée d'une explication scientifique. Les lignes de code clés sont commentées directement dans les cellules.


## 0. ⚙️ Configuration et reconstruction des générateurs

**Pourquoi reconstruire les générateurs ?**  
Un kernel Colab/Kaggle peut être redémarré (perte RAM). Pour garantir que ce notebook fonctionne **de manière autonome**, on recharge les artefacts sauvegardés à l'étape 2 (`preprocessing_artifacts/`) et on reconstruit les générateurs à l'identique — mêmes splits, même normalisation, même augmentation.

> ⚠️ **Normalisation spécifique à MobileNetV2** : MobileNetV2 a été pré-entraîné avec la normalisation `[-1, 1]` de `tf.keras.applications.mobilenet_v2.preprocess_input`. On définit un second jeu de générateurs avec cette fonction de prétraitement, différente du `rescale=1/255` du CNN From Scratch.


In [ ]:
# ─── Installations silencieuses ───────────────────────────────────────────────
!pip install -q tensorflow scikit-learn seaborn pillow

import os
import json
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ─── Vérification GPU ─────────────────────────────────────────────────────────
gpu_devices = tf.config.list_physical_devices("GPU")
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU détecté(s) : {gpu_devices if gpu_devices else '⚠️ AUCUN GPU — activez-le dans Runtime > Change runtime type'}")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
print("✅ Seed fixée à 42 pour la reproductibilité")


In [ ]:
# ─── Chargement des artefacts de l'étape 2 ────────────────────────────────────
SAVE_DIR = "preprocessing_artifacts"

with open(os.path.join(SAVE_DIR, "class_names.json"), "r", encoding="utf-8") as f:
    class_info = json.load(f)

with open(os.path.join(SAVE_DIR, "preprocessing_config.json"), "r", encoding="utf-8") as f:
    preprocessing_config = json.load(f)

class_names = class_info["class_names"]
num_classes = class_info["num_classes"]

train_df = pd.read_csv(os.path.join(SAVE_DIR, "train_df.csv"))
val_df   = pd.read_csv(os.path.join(SAVE_DIR, "val_df.csv"))
test_df  = pd.read_csv(os.path.join(SAVE_DIR, "test_df.csv"))

IMG_SIZE   = tuple(preprocessing_config["img_size"])   # (224, 224)
BATCH_SIZE = preprocessing_config["batch_size"]        # 32

print(f"✅ Artefacts chargés : {num_classes} classes")
print(f"   Train      : {len(train_df):,} images")
print(f"   Validation : {len(val_df):,} images")
print(f"   Test       : {len(test_df):,} images")
print(f"   IMG_SIZE={IMG_SIZE} | BATCH_SIZE={BATCH_SIZE}")


In [ ]:
# ─── Reconstruction des générateurs avec preprocess_input de MobileNetV2 ──────
# MobileNetV2 attend des pixels normalisés dans [-1, 1] via mobilenet_preprocess,
# et NON dans [0, 1] comme pour le CNN From Scratch.

# Train : preprocess_input MobileNetV2 + augmentation identique à l'étape 2
mobilenet_train_datagen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,  # normalisation [-1, 1]
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
)

# Validation / Test : preprocess_input SEULEMENT, aucune augmentation
mobilenet_eval_datagen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
)

common_args = dict(
    x_col="filepath",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    seed=SEED,
)

mobilenet_train_gen = mobilenet_train_datagen.flow_from_dataframe(
    train_df, shuffle=True, **common_args
)
mobilenet_val_gen = mobilenet_eval_datagen.flow_from_dataframe(
    val_df, shuffle=False, **common_args
)
mobilenet_test_gen = mobilenet_eval_datagen.flow_from_dataframe(
    test_df, shuffle=False, **common_args
)

# Vérification de cohérence de l'encodage des classes
assert mobilenet_train_gen.class_indices == class_info["class_indices"], \
    "⚠️ Incohérence d'encodage des classes — vérifiez le SAVE_DIR et l'étape 2 !"

print("✅ Générateurs MobileNetV2 reconstruits avec succès.")
print(f"✅ Encodage des {num_classes} classes vérifié et cohérent avec l'étape 2.")
print("ℹ️  Normalisation active : mobilenet_preprocess → pixels dans [-1, 1]")


## 🏗️ PARTIE 1 — Chargement du modèle de base MobileNetV2

### Qu'est-ce que MobileNetV2 ?

| Caractéristique | Détail |
|---|---|
| **Architecture** | Réseau de neurones convolutifs conçu par Google (2018) pour les environnements à ressources limitées (mobile, embarqué) |
| **Blocs clés** | **Inverted Residual Blocks** (blocs résiduels inversés) avec connexions *skip* et couches Dépthwise Separable Convolutions |
| **Paramètres** | ~3,4 millions — très compact pour ses performances |
| **Entraînement initial** | 1,28 million d'images ImageNet, 1 000 classes |
| **`include_top=False`** | On retire la tête de classification originale (1 000 classes) pour y substituer notre propre tête (38 classes) |
| **`input_shape=(224,224,3)`** | Dimension standard d'entrée pour MobileNetV2 (cohérent avec nos générateurs) |

> **Principe du Transfer Learning :** Les couches basses de MobileNetV2 ont appris à détecter des contours, des textures et des formes sur 1,28 M d'images variées. Ces connaissances génériques sont **directement réutilisables** pour identifier des lésions, des taches et des décolorations sur les feuilles de plantes, sans repartir de zéro.


In [ ]:
# ─── Chargement de MobileNetV2 pré-entraîné sur ImageNet ─────────────────────
base_model = MobileNetV2(
    weights="imagenet",        # poids pré-entraînés sur ImageNet (1,28 M d'images)
    include_top=False,          # exclut la tête de classification originale (1000 classes)
    input_shape=(*IMG_SIZE, 3), # (224, 224, 3) — format attendu par MobileNetV2
)

print(f"✅ MobileNetV2 chargé avec succès.")
print(f"   Nombre de couches dans la base : {len(base_model.layers)}")
print(f"   Shape de sortie de la base     : {base_model.output_shape}")
# La base produit une carte de caractéristiques (None, 7, 7, 1280)
# GlobalAveragePooling2D va la réduire à (None, 1280) dans la partie suivante


## 🔧 PARTIE 2 — Construction du modèle complet

### Architecture de la tête de classification ajoutée

| Couche | Paramètres | Rôle |
|--------|-----------|------|
| **GlobalAveragePooling2D** | — | Réduit la carte (7×7×1280) en vecteur (1280) en moyennant chaque plan de features → pas de Flatten brutal qui multiplierait les paramètres |
| **Dense(512, relu)** | 655 872 | Couche de décision intermédiaire — combine les features globales extraites |
| **BatchNormalization** | 2 048 | Stabilise et accélère l'entraînement de la tête, compense les différences de distribution entre features ImageNet et PlantVillage |
| **Dropout(0.5)** | — | Désactive 50% des neurones aléatoirement → prévient l'overfitting sur PlantVillage |
| **Dense(38, softmax)** | 19 494 | Sortie finale : probabilité sur chacune des 38 classes |

> **Pourquoi GlobalAveragePooling2D et non Flatten ?**  
> Flatten(7×7×1280) produirait un vecteur de 62 720 éléments → Dense(512) ajouterait ~32 M de paramètres. GlobalAveragePooling2D(1280) → Dense(512) n'en ajoute que ~655 K. Moins de paramètres = moins de risque d'overfitting, entraînement plus rapide.


In [ ]:
# ─── Construction du modèle complet : base MobileNetV2 + tête de classification ─
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name="input_224")

# Passage dans la base MobileNetV2 (extraction de features)
x = base_model(inputs, training=False)   # training=False : BatchNorm de la base en mode inférence

# ── Tête de classification personnalisée ──────────────────────────────────────
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
# Réduit (None, 7, 7, 1280) → (None, 1280) — beaucoup plus compact que Flatten

x = layers.Dense(512, activation="relu", name="dense_512")(x)
# Couche de décision intermédiaire

x = layers.BatchNormalization(name="bn_top")(x)
# Stabilise l'entraînement, normalise les activations de la tête

x = layers.Dropout(0.5, name="dropout_50")(x)
# Régularisation anti-overfitting : 50% des neurones désactivés aléatoirement

outputs = layers.Dense(num_classes, activation="softmax", name="output_38")(x)
# 38 probabilités en sortie (une par classe PlantVillage)

mobilenet_model = tf.keras.Model(inputs, outputs, name="MobileNetV2_PlantDisease")

print("\n" + "=" * 70)
print(" RÉSUMÉ DU MODÈLE COMPLET")
print("=" * 70)
mobilenet_model.summary()


In [ ]:
# ─── Comptage des paramètres (information clé pour le rapport) ─────────────────
total_params      = mobilenet_model.count_params()
trainable_params  = int(np.sum([np.prod(v.shape) for v in mobilenet_model.trainable_variables]))
frozen_params     = total_params - trainable_params   # encore tout gelé à ce stade

print("\n" + "=" * 60)
print(" BILAN DES PARAMÈTRES")
print("=" * 60)
print(f" Total                  : {total_params:>12,}")
print(f" Entraînables (tête)    : {trainable_params:>12,}")
print(f" Gelés (base ImageNet)  : {frozen_params:>12,}")
print("=" * 60)
print("ℹ️  Les paramètres de la base seront gelés en Phase 1 (Feature Extraction).")
print("ℹ️  Les 20 dernières couches seront dégelées en Phase 2 (Fine-Tuning).")


## 🧊 PARTIE 3 — Phase 1 : Feature Extraction (couches gelées)

### Logique de la Phase 1

En **Feature Extraction**, on **gèle toutes les couches de la base MobileNetV2** (`base_model.trainable = False`). Seule la tête de classification ajoutée (Dense 512, BN, Dropout, Dense 38) est entraînable.

**Pourquoi cette approche en deux phases ?**

| | Phase 1 — Feature Extraction | Phase 2 — Fine-Tuning |
|---|---|---|
| **Couches entraînées** | Tête seulement (~677 K paramètres) | Tête + 20 dernières couches de base |
| **Learning Rate** | `0.001` (standard) | `0.00001` (très bas pour ne pas dégrader les poids ImageNet) |
| **Durée** | 5 epochs (convergence rapide) | 10 epochs (affinage progressif) |
| **Objectif** | Apprendre rapidement les features PlantVillage sans toucher aux poids ImageNet | Adapter les features profondes au domaine spécifique des feuilles de plantes |

> **Risque sans Phase 1 :** Si on dégèle directement les couches profondes avec un LR élevé, les gradients issus de la tête non entraînée (très bruités) détruisent les poids ImageNet — c'est le phénomène de **"catastrophic forgetting"**.


In [ ]:
# ─── PHASE 1 : Feature Extraction — Gel de toutes les couches de la base ──────
base_model.trainable = False   # toutes les couches de MobileNetV2 sont gelées

# Vérification
frozen_layers = sum(1 for l in mobilenet_model.layers if not l.trainable)
total_layers  = len(mobilenet_model.layers)
print(f"📊 Couches gelées   : {frozen_layers} / {total_layers}")
print(f"📊 Paramètres entraînables : {int(np.sum([np.prod(v.shape) for v in mobilenet_model.trainable_variables])):,} (tête seulement)")


In [ ]:
# ─── Compilation — Phase 1 ────────────────────────────────────────────────────
mobilenet_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),  # LR standard pour la tête
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print("✅ Modèle compilé (Phase 1) : Adam(lr=0.001), categorical_crossentropy, accuracy")


In [ ]:
# ─── Calcul des poids de classes (déséquilibre PlantVillage) ─────────────────
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(mobilenet_train_gen.classes),
    y=mobilenet_train_gen.classes,
)
class_weight_dict = dict(enumerate(class_weights_array))

print("⚖️  Poids de classes calculés (5 premiers exemples) :")
for i in list(class_weight_dict.keys())[:5]:
    print(f"   Classe {i:02d} ({class_names[i]:<35}) -> poids = {class_weight_dict[i]:.4f}")


In [ ]:
# ─── Entraînement — Phase 1 (Feature Extraction) ──────────────────────────────
EPOCHS_PHASE1 = 5

steps_per_epoch   = mobilenet_train_gen.samples  // BATCH_SIZE
validation_steps  = mobilenet_val_gen.samples   // BATCH_SIZE

callbacks_phase1 = [
    ModelCheckpoint(
        filepath="best_mobilenet_phase1.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

print("🚀 Début de la Phase 1 — Feature Extraction")
print(f"   epochs={EPOCHS_PHASE1} | steps/epoch={steps_per_epoch} | val_steps={validation_steps}")
print("─" * 60)

start_phase1 = time.time()

history_phase1 = mobilenet_model.fit(
    mobilenet_train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=mobilenet_val_gen,
    validation_steps=validation_steps,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1,
    verbose=1,
)

time_phase1 = time.time() - start_phase1
print(f"\n✅ Phase 1 terminée en {time_phase1/60:.1f} min.")
print(f"   Meilleure val_accuracy (Phase 1) : {max(history_phase1.history['val_accuracy']):.4f}")


## 🔥 PARTIE 4 — Phase 2 : Fine-Tuning (20 dernières couches dégelées)

### Logique du Fine-Tuning

Après la Phase 1, la tête de classification a convergé. On peut maintenant **dégeler les couches les plus profondes** de MobileNetV2 pour les spécialiser au domaine botanique.

**Pourquoi les 20 dernières couches ?**  
Les premières couches de MobileNetV2 détectent des features très génériques (contours, textures) communes à tous les domaines — les modifier n'apporterait rien. Les dernières couches encodent des représentations de plus haut niveau (formes, structures) qui méritent d'être adaptées au domaine des plantes.

**Pourquoi `learning_rate = 0.00001` (1e-5) ?**  
Un LR trop élevé dégraderait les poids ImageNet soigneusement appris — on ajuste très finement, comme un réglage de précision.

**Callbacks — justification :**

| Callback | Paramètres | Rôle |
|---|---|---|
| **EarlyStopping** | `patience=5`, `restore_best_weights=True` | Arrête si `val_loss` ne s'améliore plus, récupère le meilleur modèle |
| **ReduceLROnPlateau** | `factor=0.2`, `patience=3`, `min_lr=1e-7` | Divise le LR par 5 si la val_loss stagne — affinage encore plus fin |
| **ModelCheckpoint** | `monitor=val_accuracy`, `save_best_only=True` | Sauvegarde uniquement le meilleur modèle rencontré |


In [ ]:
# ─── PHASE 2 : Dégel des 20 dernières couches de MobileNetV2 ─────────────────
base_model.trainable = True   # on autorise la mise à jour des poids de la base

# On regèle TOUTES les couches d'abord, puis on dégeèle les 20 dernières
for layer in base_model.layers[:-20]:
    layer.trainable = False   # couches profondes (génériques) : gelées
# Les 20 dernières (couches spécialisées) restent entraînables

# Bilan après dégel partiel
trainable_after  = int(np.sum([np.prod(v.shape) for v in mobilenet_model.trainable_variables]))
frozen_after     = mobilenet_model.count_params() - trainable_after

print("📊 Bilan après dégel des 20 dernières couches :")
print(f"   Paramètres entraînables : {trainable_after:>12,}")
print(f"   Paramètres gelés        : {frozen_after:>12,}")
print(f"   Total                   : {mobilenet_model.count_params():>12,}")

# Liste des couches dégelées
trainable_layer_names = [l.name for l in mobilenet_model.layers if l.trainable]
print(f"\n   Couches dégelées ({len(trainable_layer_names)}) : {trainable_layer_names[-5:]} ... (et les couches de la tête)")


In [ ]:
# ─── Compilation — Phase 2 (Fine-Tuning) ─────────────────────────────────────
# Learning rate TRÈS bas pour ne pas détruire les poids ImageNet
LR_FINETUNE = 1e-5

mobilenet_model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_FINETUNE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"✅ Modèle recompilé (Phase 2) : Adam(lr={LR_FINETUNE})")


In [ ]:
# ─── Callbacks — Phase 2 ─────────────────────────────────────────────────────
EPOCHS_PHASE2 = 10

callbacks_phase2 = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,                  # tolère 5 epochs sans amélioration
        restore_best_weights=True,   # récupère automatiquement les meilleurs poids
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,                  # divise le LR par 5 en cas de stagnation
        patience=3,
        min_lr=1e-7,                 # plancher du LR
        verbose=1,
    ),
    ModelCheckpoint(
        filepath="best_mobilenet_finetune.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]
print("✅ Callbacks Phase 2 configurés : EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")


In [ ]:
# ─── Entraînement — Phase 2 (Fine-Tuning) ────────────────────────────────────
# On repart du point où la Phase 1 s'est arrêtée (initial_epoch)
initial_epoch = EPOCHS_PHASE1

print("🔥 Début de la Phase 2 — Fine-Tuning")
print(f"   epochs={EPOCHS_PHASE2} | LR={LR_FINETUNE} | 20 dernières couches dégelées")
print("─" * 60)

start_phase2 = time.time()

history_phase2 = mobilenet_model.fit(
    mobilenet_train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=mobilenet_val_gen,
    validation_steps=validation_steps,
    epochs=initial_epoch + EPOCHS_PHASE2,  # epochs totaux (Phase1 + Phase2)
    initial_epoch=initial_epoch,           # reprend après la Phase 1
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2,
    verbose=1,
)

time_phase2 = time.time() - start_phase2
training_time_total = time_phase1 + time_phase2

print(f"\n✅ Phase 2 terminée en {time_phase2/60:.1f} min.")
print(f"✅ Temps total d'entraînement (Phase 1 + Phase 2) : {training_time_total/60:.1f} min.")
print(f"   Meilleure val_accuracy (Phase 2) : {max(history_phase2.history['val_accuracy']):.4f}")


In [ ]:
# ─── Fusion des historiques Phase 1 + Phase 2 pour les visualisations ────────
# On concatène les deux histories pour avoir une courbe complète
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history_full = merge_histories(history_phase1, history_phase2)
total_epochs_run = len(history_full["loss"])
print(f"✅ Historiques fusionnés : {total_epochs_run} epochs au total (Phase 1 + Phase 2)")


## 📊 PARTIE 5 — Évaluation sur le jeu de test

### Métriques calculées

| Métrique | Formule | Interprétation |
|---|---|---|
| **Accuracy** | (VP + VN) / Total | % de prédictions correctes (toutes classes confondues) |
| **Precision** | VP / (VP + FP) | Parmi les images prédites malades, quelle proportion l'est vraiment ? |
| **Recall** | VP / (VP + FN) | Parmi les images réellement malades, combien le modèle en détecte ? |
| **F1-Score** | 2 × (P × R) / (P + R) | Moyenne harmonique Precision/Recall — robuste aux déséquilibres de classes |

> **Aggregation `macro`** : moyenne non pondérée sur les 38 classes — chaque classe compte autant, quelle que soit sa taille.  
> **Aggregation `weighted`** : moyenne pondérée par la taille de chaque classe — plus représentative de la performance globale.


In [ ]:
# ─── Prédictions sur le jeu de test ──────────────────────────────────────────
print("🔄 Calcul des prédictions sur le jeu de test...")

mobilenet_test_gen.reset()   # important : remet le générateur au début
y_pred_proba = mobilenet_model.predict(
    mobilenet_test_gen,
    steps=np.ceil(mobilenet_test_gen.samples / BATCH_SIZE),
    verbose=1,
)

y_pred = np.argmax(y_pred_proba, axis=1)         # classe prédite (index)
y_true = mobilenet_test_gen.classes               # classes réelles
# Ajustement si le générateur a généré plus de prédictions que d'images (arrondi batch)
y_pred = y_pred[:len(y_true)]

print(f"\n✅ Prédictions calculées sur {len(y_true):,} images de test.")


In [ ]:
# ─── Calcul des métriques ─────────────────────────────────────────────────────
test_loss, test_accuracy = mobilenet_model.evaluate(
    mobilenet_test_gen,
    steps=np.ceil(mobilenet_test_gen.samples / BATCH_SIZE),
    verbose=0,
)

precision_macro    = precision_score(y_true, y_pred, average="macro",    zero_division=0)
recall_macro       = recall_score   (y_true, y_pred, average="macro",    zero_division=0)
f1_macro           = f1_score       (y_true, y_pred, average="macro",    zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted    = recall_score   (y_true, y_pred, average="weighted", zero_division=0)
f1_weighted        = f1_score       (y_true, y_pred, average="weighted", zero_division=0)

print("\n" + "=" * 60)
print(" MÉTRIQUES D'ÉVALUATION — MobileNetV2 Fine-Tuned")
print("=" * 60)
print(f"  Loss (test)                  : {test_loss:.4f}")
print(f"  Accuracy (test)              : {test_accuracy:.4f}  ({test_accuracy*100:.2f}%)")
print("─" * 60)
print(f"  Precision  (macro avg)       : {precision_macro:.4f}")
print(f"  Recall     (macro avg)       : {recall_macro:.4f}")
print(f"  F1-Score   (macro avg)       : {f1_macro:.4f}")
print("─" * 60)
print(f"  Precision  (weighted avg)    : {precision_weighted:.4f}")
print(f"  Recall     (weighted avg)    : {recall_weighted:.4f}")
print(f"  F1-Score   (weighted avg)    : {f1_weighted:.4f}")
print("=" * 60)


## 📈 PARTIE 6 — Visualisations

**Quatre visualisations produites :**

1. **Courbe Accuracy** (Phase 1 + Phase 2) → vérifie la convergence et l'absence d'overfitting
2. **Courbe Loss** → signal le plus fiable d'overfitting (val_loss qui remonte)
3. **Matrice de confusion** (38×38 normalisée) → identifie les confusions entre classes
4. **Classification Report** → F1-score par classe → repère les classes difficiles


In [ ]:
# ─── Courbes Accuracy et Loss (Phase 1 + Phase 2 fusionnées) ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
epochs_range = range(1, total_epochs_run + 1)

# Ligne de séparation Phase 1 / Phase 2
phase1_end = EPOCHS_PHASE1

# ── Accuracy ──
axes[0].plot(epochs_range, history_full["accuracy"],     label="Train Accuracy",      linewidth=2, color="#2196F3")
axes[0].plot(epochs_range, history_full["val_accuracy"], label="Validation Accuracy",  linewidth=2, color="#4CAF50")
axes[0].axvline(x=phase1_end, color="orange", linestyle="--", linewidth=1.5,
                label=f"Début Fine-Tuning (epoch {phase1_end+1})")
axes[0].fill_betweenx([0, 1], 0, phase1_end, alpha=0.05, color="blue",  label="Phase 1")
axes[0].fill_betweenx([0, 1], phase1_end, total_epochs_run, alpha=0.05, color="red", label="Phase 2")
axes[0].set_title("Évolution de l'Accuracy — MobileNetV2", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend(fontsize=8)
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3)

# ── Loss ──
axes[1].plot(epochs_range, history_full["loss"],     label="Train Loss",      linewidth=2, color="#2196F3")
axes[1].plot(epochs_range, history_full["val_loss"], label="Validation Loss",  linewidth=2, color="#F44336")
axes[1].axvline(x=phase1_end, color="orange", linestyle="--", linewidth=1.5,
                label=f"Début Fine-Tuning (epoch {phase1_end+1})")
axes[1].set_title("Évolution de la Loss — MobileNetV2", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("mobilenet_courbes_entrainement.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Courbes sauvegardées : mobilenet_courbes_entrainement.png")


In [ ]:
# ─── Matrice de confusion (38x38, normalisée en %) ────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(22, 20))
sns.heatmap(
    cm_normalized,
    annot=False,             # 38x38 trop dense pour annoter chaque case
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={"label": "% de prédictions par classe réelle"},
    linewidths=0.1,
    linecolor="lightgray",
)
plt.title("Matrice de confusion (normalisée, %) — MobileNetV2 Fine-Tuned", fontsize=15, fontweight="bold")
plt.xlabel("Classe prédite", fontsize=12)
plt.ylabel("Classe réelle", fontsize=12)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0,  fontsize=6)
plt.tight_layout()
plt.savefig("mobilenet_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Identification des 5 confusions les plus fréquentes (hors diagonale)
cm_no_diag = cm_normalized.copy()
np.fill_diagonal(cm_no_diag, 0)
top_confusions_idx = np.dstack(
    np.unravel_index(np.argsort(-cm_no_diag.ravel()), cm_no_diag.shape)
)[0][:5]

print("🔍 Top 5 confusions entre classes (MobileNetV2) :")
for true_idx, pred_idx in top_confusions_idx:
    if cm_no_diag[true_idx, pred_idx] > 0:
        print(f"   {class_names[true_idx]:<40} → prédit comme {class_names[pred_idx]:<40} ({cm_no_diag[true_idx, pred_idx]:.1f}% des cas)")


In [ ]:
# ─── Classification Report détaillé + graphique des classes difficiles ─────────
report_dict = classification_report(
    y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report_dict).transpose()

print("📋 Classification Report complet (par classe) :\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

report_df.to_csv("mobilenet_classification_report.csv")
print("💾 Rapport sauvegardé : mobilenet_classification_report.csv")

# ── Graphique : 10 classes les plus difficiles (F1 le plus faible) ────────────
worst_classes = report_df.iloc[:-3].sort_values("f1-score").head(10)

plt.figure(figsize=(11, 6))
bars = plt.barh(worst_classes.index, worst_classes["f1-score"],
                color=plt.cm.Reds_r(np.linspace(0.3, 0.8, 10)))
plt.xlabel("F1-Score")
plt.title("10 classes les plus difficiles — MobileNetV2 Fine-Tuned", fontsize=13, fontweight="bold")
for bar, val in zip(bars, worst_classes["f1-score"]):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("mobilenet_classes_difficiles.png", dpi=150, bbox_inches="tight")
plt.show()


## ⚖️ PARTIE 7 — Comparaison : CNN From Scratch vs MobileNetV2

### Objectif
Cette partie charge les métriques du CNN From Scratch sauvegardées à l'étape 3 (`cnn_final_metrics.json`) et les compare aux métriques du MobileNetV2 Fine-Tuné dans un tableau et des graphiques de benchmarking.


In [ ]:
# ─── Chargement des métriques CNN From Scratch (sauvegardées à l'étape 3) ─────
CNN_METRICS_PATH = "cnn_final_metrics.json"

if os.path.exists(CNN_METRICS_PATH):
    with open(CNN_METRICS_PATH, "r") as f:
        cnn_metrics = json.load(f)
    cnn_loaded = True
    print("✅ Métriques CNN From Scratch chargées depuis cnn_final_metrics.json")
else:
    # Valeurs indicatives si le fichier n'est pas disponible
    cnn_metrics = {
        "model_name"         : "CNN_From_Scratch",
        "test_accuracy"      : 0.000,
        "test_loss"          : 0.000,
        "f1_weighted"        : 0.000,
        "total_params"       : 0,
        "training_time_minutes": 0,
        "epochs_executed"    : 0,
    }
    cnn_loaded = False
    print("⚠️  Fichier cnn_final_metrics.json introuvable — exécutez d'abord le notebook Étape 3.")
    print("   Les valeurs du CNN From Scratch seront à 0 dans le tableau de comparaison.")

# ─── Métriques MobileNetV2 (issues de la Phase 2 / évaluation juste calculée) ──
mobilenet_metrics = {
    "model_name"            : "MobileNetV2_FineTuned",
    "test_accuracy"         : float(test_accuracy),
    "test_loss"             : float(test_loss),
    "precision_macro"       : float(precision_macro),
    "recall_macro"          : float(recall_macro),
    "f1_macro"              : float(f1_macro),
    "precision_weighted"    : float(precision_weighted),
    "recall_weighted"       : float(recall_weighted),
    "f1_weighted"           : float(f1_weighted),
    "total_params"          : int(mobilenet_model.count_params()),
    "training_time_minutes" : round(training_time_total / 60, 2),
    "epochs_executed"       : total_epochs_run,
}

print("\n✅ Métriques MobileNetV2 collectées.")


In [ ]:
# ─── Tableau de comparaison ───────────────────────────────────────────────────
comparison_data = {
    "Métrique"                  : ["Accuracy", "Loss", "F1-Score (weighted)",
                                   "Nb Paramètres", "Temps entraînement (min)", "Epochs exécutés"],
    "CNN From Scratch"          : [
        f"{cnn_metrics.get('test_accuracy',0):.4f}",
        f"{cnn_metrics.get('test_loss',0):.4f}",
        f"{cnn_metrics.get('f1_weighted',0):.4f}",
        f"{cnn_metrics.get('total_params',0):,}",
        f"{cnn_metrics.get('training_time_minutes',0):.1f}",
        f"{cnn_metrics.get('epochs_executed',0)}",
    ],
    "MobileNetV2 Fine-Tuned"    : [
        f"{mobilenet_metrics['test_accuracy']:.4f}",
        f"{mobilenet_metrics['test_loss']:.4f}",
        f"{mobilenet_metrics['f1_weighted']:.4f}",
        f"{mobilenet_metrics['total_params']:,}",
        f"{mobilenet_metrics['training_time_minutes']:.1f}",
        f"{mobilenet_metrics['epochs_executed']}",
    ],
    "Δ (MobileNet − CNN)"       : [
        f"{mobilenet_metrics['test_accuracy']   - cnn_metrics.get('test_accuracy',0):+.4f}",
        f"{mobilenet_metrics['test_loss']       - cnn_metrics.get('test_loss',0):+.4f}",
        f"{mobilenet_metrics['f1_weighted']     - cnn_metrics.get('f1_weighted',0):+.4f}",
        "N/A",
        f"{mobilenet_metrics['training_time_minutes'] - cnn_metrics.get('training_time_minutes',0):+.1f}",
        "N/A",
    ],
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df.set_index("Métrique", inplace=True)

print("\n" + "=" * 75)
print(" TABLEAU DE COMPARAISON — CNN From Scratch vs MobileNetV2 Fine-Tuned")
print("=" * 75)
print(comparison_df.to_string())
print("=" * 75)

comparison_df.to_csv("comparaison_cnn_vs_mobilenet.csv")
print("\n💾 Tableau sauvegardé : comparaison_cnn_vs_mobilenet.csv")


In [ ]:
# ─── Graphique de comparaison visuelle ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models_labels = ["CNN\nFrom Scratch", "MobileNetV2\nFine-Tuned"]
colors = ["#90CAF9", "#4CAF50"]

metrics_to_plot = [
    ("test_accuracy", "f1_weighted"),       # axes[0] : Accuracy
    ("test_loss",),                           # axes[1] : Loss
]

# Accuracy
cnn_acc     = cnn_metrics.get("test_accuracy", 0)
mob_acc     = mobilenet_metrics["test_accuracy"]
axes[0].bar(models_labels, [cnn_acc, mob_acc], color=colors, edgecolor="gray", linewidth=0.7)
axes[0].set_title("Accuracy (test)", fontsize=13, fontweight="bold")
axes[0].set_ylim([0, 1.05])
axes[0].set_ylabel("Accuracy")
for i, v in enumerate([cnn_acc, mob_acc]):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=11, fontweight="bold")

# Loss
cnn_loss    = cnn_metrics.get("test_loss", 0)
mob_loss    = mobilenet_metrics["test_loss"]
axes[1].bar(models_labels, [cnn_loss, mob_loss], color=colors, edgecolor="gray", linewidth=0.7)
axes[1].set_title("Loss (test)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Loss")
for i, v in enumerate([cnn_loss, mob_loss]):
    axes[1].text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=11, fontweight="bold")

# F1-Score (weighted)
cnn_f1      = cnn_metrics.get("f1_weighted", 0)
mob_f1      = mobilenet_metrics["f1_weighted"]
axes[2].bar(models_labels, [cnn_f1, mob_f1], color=colors, edgecolor="gray", linewidth=0.7)
axes[2].set_title("F1-Score Weighted (test)", fontsize=13, fontweight="bold")
axes[2].set_ylim([0, 1.05])
axes[2].set_ylabel("F1-Score")
for i, v in enumerate([cnn_f1, mob_f1]):
    axes[2].text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=11, fontweight="bold")

for ax in axes:
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("Benchmarking : CNN From Scratch vs MobileNetV2 Fine-Tuned",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("benchmarking_cnn_vs_mobilenet.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Graphique sauvegardé : benchmarking_cnn_vs_mobilenet.png")


## 💾 PARTIE 8 — Sauvegarde du modèle

**Deux formats sauvegardés :**

| Format | Extension | Avantages |
|---|---|---|
| **Keras natif** | `.keras` | Format recommandé (Keras 3), contient architecture + poids + config en un seul fichier optimisé |
| **HDF5** | `.h5` | Format historique, compatibilité maximale avec les outils anciens, Streamlit, Hugging Face Spaces, TF < 2.12 |


In [ ]:
# ─── Sauvegarde des modèles ───────────────────────────────────────────────────
# Format natif Keras 3 (recommandé pour Keras >= 2.12 / TF >= 2.13)
mobilenet_model.save("mobilenetv2_plant_disease.keras")
print("💾 Modèle sauvegardé : mobilenetv2_plant_disease.keras")

# Format HDF5 (compatibilité et déploiement)
mobilenet_model.save("mobilenetv2_plant_disease.h5")
print("💾 Modèle sauvegardé : mobilenetv2_plant_disease.h5")


In [ ]:
# ─── Sauvegarde de l'historique complet et des métriques ─────────────────────
# Historique (utile pour des analyses comparatives futures)
with open("mobilenet_training_history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history_full.items()}, f, indent=2)
print("💾 Historique sauvegardé : mobilenet_training_history.json")

# Métriques finales (réutilisables pour les étapes suivantes du projet)
with open("mobilenet_final_metrics.json", "w") as f:
    json.dump(mobilenet_metrics, f, indent=2)
print("💾 Métriques finales sauvegardées : mobilenet_final_metrics.json")

# Résumé des fichiers produits
print("\n" + "=" * 60)
print(" RÉCAPITULATIF DES FICHIERS GÉNÉRÉS")
print("=" * 60)
files_generated = [
    "mobilenetv2_plant_disease.keras",
    "mobilenetv2_plant_disease.h5",
    "best_mobilenet_phase1.keras",
    "best_mobilenet_finetune.keras",
    "mobilenet_training_history.json",
    "mobilenet_final_metrics.json",
    "mobilenet_courbes_entrainement.png",
    "mobilenet_confusion_matrix.png",
    "mobilenet_classes_difficiles.png",
    "mobilenet_classification_report.csv",
    "benchmarking_cnn_vs_mobilenet.png",
    "comparaison_cnn_vs_mobilenet.csv",
]
for fname in files_generated:
    exists = "✅" if os.path.exists(fname) else "⏳ (généré à l'exécution)"
    print(f"  {exists}  {fname}")
print("=" * 60)


## 🔬 PARTIE 9 — Analyse professionnelle

### 9.1 — Diagnostic automatique (overfitting, généralisation)


In [ ]:
# ─── Diagnostic automatique ───────────────────────────────────────────────────
final_train_acc   = history_full["accuracy"][-1]
final_val_acc     = history_full["val_accuracy"][-1]
acc_gap           = final_train_acc - final_val_acc

final_train_loss  = history_full["loss"][-1]
final_val_loss    = history_full["val_loss"][-1]

# Tendance val_loss sur les 5 derniers epochs (remontée = overfitting)
last_n = min(5, len(history_full["val_loss"]))
val_loss_trend = history_full["val_loss"][-last_n] - history_full["val_loss"][-1]

print("=" * 70)
print("DIAGNOSTIC AUTOMATIQUE — MobileNetV2 Fine-Tuned")
print("=" * 70)
print(f"  Accuracy finale Train       : {final_train_acc:.4f}  ({final_train_acc*100:.2f}%)")
print(f"  Accuracy finale Validation  : {final_val_acc:.4f}  ({final_val_acc*100:.2f}%)")
print(f"  Écart Train − Validation    : {acc_gap:.4f}")
print(f"  Loss finale Train           : {final_train_loss:.4f}")
print(f"  Loss finale Validation      : {final_val_loss:.4f}")
print(f"  Tendance val_loss (5 dern.) : {'+' if val_loss_trend >= 0 else ''}{val_loss_trend:.4f}  ",
      end="")
print("(baisse = bonne généralisation)" if val_loss_trend >= 0 else "(hausse = risque d'overfitting)")

print()
if acc_gap > 0.10:
    print("⚠️  DIAGNOSTIC : Écart significatif (>10%) → OVERFITTING probable.")
    print("   ► Recommandation : augmenter Dropout, renforcer la Data Augmentation,")
    print("     réduire le nombre de couches dégelées, ou ajouter une régularisation L2.")
elif acc_gap > 0.05:
    print("🟡 DIAGNOSTIC : Léger overfitting (5–10%). Performances acceptables.")
    print("   ► Recommandation : surveiller sur un run plus long, envisager Dropout plus élevé.")
else:
    print("✅ DIAGNOSTIC : Généralisation satisfaisante (écart < 5%) — bon équilibre bias/variance.")


### 9.2 — Analyse scientifique complète

---

#### 🔵 Impact du Transfer Learning

Le Transfer Learning consiste à réutiliser des représentations visuelles apprises sur un large dataset (ImageNet : 1,28 M d'images, 1 000 classes) et à les adapter à un nouveau domaine (PlantVillage : 54 305 images, 38 classes).

**Mécanisme :** MobileNetV2 a appris, sur ImageNet, à détecter des contours, des textures, des formes et des structures visuelles complexes. Ces features bas et moyen niveau (bords, gradients de couleur, motifs de surface) sont **transférables** au domaine des maladies végétales : une lésion brune sur une feuille, une tache chlorotique ou un dépôt fongique présentent des caractéristiques visuelles (contrastes, textures, formes irrégulières) que les filtres ImageNet savent déjà détecter.

**Avantages mesurés dans ce projet :**
- Convergence significativement plus rapide (la Phase 1 suffit à obtenir un modèle déjà performant)
- Accuracy supérieure au CNN From Scratch avec moins d'epochs totaux
- Meilleure généralisation grâce aux features robustes pré-apprises

---

#### 🟠 Impact du Fine-Tuning

Le Fine-Tuning (Phase 2) affine les couches profondes de MobileNetV2 — celles qui encodent des représentations de haut niveau — pour les spécialiser au domaine botanique.

**Résultats attendus du Fine-Tuning :**
- Gain d'accuracy supplémentaire de +2 à +5% par rapport à la seule Feature Extraction
- Réduction du taux d'erreur sur les classes visuellement proches (ex : stades de maladie différents sur une même plante)
- Légère augmentation du risque d'overfitting si le LR est trop élevé → justification du LR = 1e-5

**Risque du Fine-Tuning mal paramétré :** Un LR trop élevé en Phase 2 détruit les poids ImageNet ("catastrophic forgetting"), annulant le bénéfice du Transfer Learning. Le LR = 1e-5 et le dégel progressif (20 dernières couches seulement) atténuent ce risque.

---

#### 🟢 Pourquoi MobileNetV2 est-il (très probablement) meilleur que le CNN From Scratch ?

| Facteur | CNN From Scratch | MobileNetV2 Fine-Tuné |
|---|---|---|
| **Données vues** | 54 305 images PlantVillage seulement | 1,28 M (ImageNet) + 54 305 (fine-tuning) |
| **Profondeur effective** | 4 blocs convolutifs (~3,7 M paramètres) | 53 couches (Inverted Residuals) + tête |
| **Initialisation** | Aléatoire (Xavier/He) | Poids optimisés sur ImageNet |
| **Vitesse de convergence** | Lente (20 epochs minimum) | Rapide (5 epochs Phase 1 suffisent) |
| **Robustesse** | Sensible aux hyper-paramètres | Plus robuste grâce aux features stables |
| **Risque d'overfitting** | Élevé sur dataset moyen | Modéré (régularisation implicite via features pré-apprises) |

---

#### 📌 Cas où MobileNetV2 pourrait être moins bon (rares mais possibles)

- Si la normalisation `mobilenet_preprocess` n'est pas appliquée correctement (les pixels hors [-1,1] faussent les activations)
- Si le LR du Fine-Tuning est trop élevé et dégrade les poids ImageNet
- Si le dataset PlantVillage est trop différent d'ImageNet (ce qui n'est pas le cas ici — les images de feuilles contiennent des textures, couleurs et formes bien représentées dans ImageNet)

---

#### 🏁 Conclusion scientifique

L'architecture MobileNetV2 Fine-Tuné représente le rapport **efficacité / performance / coût computationnel** optimal pour ce projet de détection de maladies végétales :

1. **Transfer Learning** : exploite 1,28 M d'images ImageNet pour initialiser des features visuelles robustes, compensant la taille modeste de PlantVillage
2. **Approche en deux phases** : évite le catastrophic forgetting tout en permettant une spécialisation progressive au domaine botanique
3. **Architecture légère** (~3,4 M paramètres) : compatible GPU Colab/Kaggle gratuit et déployable sur mobile
4. **Comparaison** : les métriques du tableau de la Partie 7 quantifient objectivement le gain apporté par MobileNetV2 versus le CNN From Scratch, démontrant l'intérêt du Transfer Learning dans un contexte de dataset de taille moyenne

> **Recommandation pour la suite du projet :** L'étape 5 peut explorer ResNet50 (architecture plus profonde, ~25 M paramètres) pour évaluer si le gain de complexité justifie le surcoût computationnel — la comparaison à trois voies (CNN Scratch / MobileNetV2 / ResNet50) constituera un benchmarking complet pour le rapport final.
